# Linear Regresssion with sklearn<br>

## Analysis of European Soccer Dataset

<br>


We will be using an open dataset from the <a href="https://www.kaggle.com">Kaggle</a> site on European Soccer.

This <a href="https://www.kaggle.com/hugomathien/soccer">European Soccer Database</a> has data on more than 25,000 matches and more than 10,000 players for European professional soccer seasons from 2008 to 2016.

This data in contained in the file *database.sqlite* that is included with the zip file. This is an example of an SQL database, which we will need to read in differently than we have csv files.


### Import Libraries<br>


In [2]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

### Read Data from the SQL Database into pandas
<br>


In [3]:
# Create your SQL connection.
from google.colab import drive
drive.mount ('/content/drive')
cnx = sqlite3.connect('drive/My Drive/Notebook14/database.sqlite')
df = pd.read_sql_query("SELECT * FROM Player_Attributes", cnx)
#This should read in the player data from the database.

Mounted at /content/drive


In [4]:
#Lets inspect this dataframe first.
df.shape

(183978, 42)

In [79]:
df.columns

Index(['id', 'player_fifa_api_id', 'player_api_id', 'date', 'overall_rating',
       'potential', 'preferred_foot', 'attacking_work_rate',
       'defensive_work_rate', 'crossing', 'finishing', 'heading_accuracy',
       'short_passing', 'volleys', 'dribbling', 'curve', 'free_kick_accuracy',
       'long_passing', 'ball_control', 'acceleration', 'sprint_speed',
       'agility', 'reactions', 'balance', 'shot_power', 'jumping', 'stamina',
       'strength', 'long_shots', 'aggression', 'interceptions', 'positioning',
       'vision', 'penalties', 'marking', 'standing_tackle', 'sliding_tackle',
       'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning',
       'gk_reflexes'],
      dtype='object')

So it looks like there are 183,978 player ratings ("overall_rating"), along with a bunch of other measurements of their relevant soccer skills such as "sprint_speed", shot_power", etc.

However, there are not 183,978 unique players. We can use the .nunique method to count the number of unique entries in a dataframe column, and we can count the number of unique FIFA ID numbers:

In [80]:
df.player_fifa_api_id.nunique()

11062

There are 11062 unique players, so presumably the database contains information on players from multiple seasons.

## Exercise 1: Average number of ratings per player:
Write a line of code below to compute and print out the average number of ratings per player, rounded to the nearest integer. The answer should come out to 17.

In [81]:
### YOUR CODE HERE!
avg_ratings_per_player = df.shape[0] / df.player_fifa_api_id.nunique()
round(avg_ratings_per_player)

17

Our goal now is to use multiple linear regression to develop a model that we can use to predict a player's overall_rating based on some of the other measurements of their soccer skills. We will need to select the columns that we want to use for this; they will obviously need to be columns with numerical entries.

In [82]:
pd.set_option("display.max_columns", None) #This is so we can see all columns.
df.head()
#We will not want to include things like "attacking_work_rate" for example that are not numerical values.

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,free_kick_accuracy,long_passing,ball_control,acceleration,sprint_speed,agility,reactions,balance,shot_power,jumping,stamina,strength,long_shots,aggression,interceptions,positioning,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,71.0,70.0,45.0,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,71.0,70.0,45.0,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,63.0,41.0,45.0,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,43.0,70.0,60.0,43.0,50.0,44.0,38.0,63.0,48.0,60.0,64.0,59.0,46.0,65.0,54.0,58.0,54.0,76.0,34.0,62.0,40.0,44.0,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,43.0,70.0,60.0,43.0,50.0,44.0,38.0,63.0,48.0,60.0,64.0,59.0,46.0,65.0,54.0,58.0,54.0,76.0,34.0,62.0,40.0,44.0,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0



### Declare the Columns We Want to Use in the Model
<br>
We will use the final 33 columns of the DataFrame:

In [83]:
features = ['crossing', 'finishing', 'heading_accuracy',
       'short_passing', 'volleys', 'dribbling', 'curve', 'free_kick_accuracy',
       'long_passing', 'ball_control', 'acceleration', 'sprint_speed',
       'agility', 'reactions', 'balance', 'shot_power', 'jumping', 'stamina',
       'strength', 'long_shots', 'aggression', 'interceptions', 'positioning',
       'vision', 'penalties', 'marking', 'standing_tackle', 'sliding_tackle',
       'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning',
       'gk_reflexes']

### Specify the Prediction Target
<br>


In [84]:
target = ['overall_rating']

Clean the Data to get rid of any NaN values<br>


In [85]:
dfclean = df.dropna()

## Exercise 2: Lost Rows
Write a line of code below to compute and print out the percentage of rows that were dropped when we dropped rows containing NaN values above. The answer should come out to 1.97%.

In [86]:
### YOUR CODE HERE!
rows_dropped = ((len(df) - len(dfclean)) / len(df)) * 100
rows_dropped = round(rows_dropped, 2)  #Rounds to 2 decimal places
rows_dropped

1.97


### Put Features and Target ('overall_rating') Values into Separate Dataframes
<br>


In [87]:
X = dfclean[features]

In [88]:
y = dfclean[target]

Let's look at a row from our features dataframe:

In [89]:
X.iloc[0]

,0
crossing,49.0
finishing,44.0
heading_accuracy,71.0
short_passing,61.0
volleys,44.0
dribbling,51.0
curve,45.0
free_kick_accuracy,39.0
long_passing,64.0
ball_control,49.0


Let's also look at our target dataframe:

In [90]:
y

,overall_rating
0,67.0
1,67.0
2,62.0
3,61.0
4,61.0
...,...
183973,83.0
183974,78.0
183975,77.0
183976,78.0


### Split the Dataset into Training and Test Datasets
<br>
We can use the train_test_split routine from sklearn to do this. We will make the training dataset 2/3 (0.67) of the dataset, and the test dataset the remaining 1/3 (0.33). Normally this is done randomly, but we fix the random state below so that we will all get the same answer for comparison.


In [91]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=123)

### Now we do our Linear Regression: Fit a model to the training set.
<br>
Note that you can see <a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html">this link</a> for a complete description of all of the attributes and methods of the LinearRegression class!


In [92]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

LinearRegression()

We can now print the fitted slopes and intercept. These are available through the regressor.coef_ and regressor.intercept_ attributes returned by regressor.fit.

In [93]:
print(regressor.coef_)
print(regressor.intercept_)

[[-0.01073426  0.01198247  0.10096165  0.07375682 -0.00081805 -0.01112493
   0.01302594  0.0080077   0.0162864   0.23015851  0.04094799  0.04693668
  -0.01435821  0.28173218  0.00321989  0.02377915  0.01303836 -0.01823793
   0.06021443 -0.0197367   0.0116052   0.01860288 -0.02018549  0.00967697
   0.01651681  0.03278068  0.01280725 -0.02843731  0.20519274  0.0576742
  -0.02921995  0.07290184  0.04630632]]
[6.23820491]


## Exercise 3: Features with largest effect on ratings?
We can use the list of coefficients ("slopes") above to find out which features have the largest effect on overall rating. These will be the features with the largest coefficients (by absolute value).

Write some code to produce and print out a list containing the **three features** that have the biggest effect on overall rating. The command np.argsort("array") is key to this. It returns the array of indices that sort "array". This was command was introduced in Notebook 12 (the sample mini-project), so review that notebook first if you need to. Call this list "largestlist", we will be using it later.

You should find that the features with the largest effects are: 'reactions', 'ball_control', and 'gk_diving'.

In [96]:
# largestlist = ###YOUR CODE HERE!

#regressor.coef_ is treated as a 1D array
coef_abs_sorted_indices = np.abs(regressor.coef_).argsort().flatten()

largest_indices = coef_abs_sorted_indices[-3:][::-1]  # top 3 & reverse order to get largest first

# Map indices to feature names
largestlist = [features[i] for i in largest_indices]

print(largestlist)


['reactions', 'ball_control', 'gk_diving']


### Make Predictions on the Test Set using our Linear Regression Model
<br>


In [56]:
y_prediction = regressor.predict(X_test)
y_prediction

array([[72.91779502],
       [64.35477607],
       [65.44360049],
       ...,
       [70.63000423],
       [77.85907634],
       [72.97955266]])

### Compare means of expected target value in test set, and the predicted target value:
<br>


In [57]:
print(y_test['overall_rating'].to_numpy().mean()) #Note y_test is a DataFrame, but y_prediction is an array.
print(y_prediction.mean())

68.63914511820153
68.64476193815995


The means are close, so that is good!

### Evaluate Linear Regression Accuracy using Root Mean Square Error
<br>


In [58]:
#We can use the mean_squared_error routine to get an error estimate:
rmse = np.sqrt(mean_squared_error(y_test, y_prediction))
print (rmse)

#We can also calculate the same quantity 'from scratch' using the statistics formula for root-mean-sqaured error:
y_testarr = y_test['overall_rating'].to_numpy()
y_predictionarr = np.transpose(y_prediction)[0]

#The above lines make sure that both arrays are the same (one dimensional) shape. Check this with .shape
print (y_testarr.shape)
print (y_predictionarr.shape)

rmse2 = np.sqrt(np.sum((y_predictionarr-y_testarr)**2)/np.size(y_testarr))
print (rmse2)

3.2746097962127103
(59517,)
(59517,)
3.2746097962127103


So our typical prediction of a player's rating is off by about 3 points from their true rating. Is this good? Let's compare it to the standard deviation of the overall rating. If we just picked some other overall rating at random as our prediction, then we would be off by about that amount.

In [59]:
y['overall_rating'].std()

7.02795002460881

Our root-mean-squared error is much less than this, so that is good.

## Exercise 4

Now fit a new linear regression model using only the three features with the largest effect that we identified in Exercise 3 above. In **train_test_split** use the same **test_size** and **random_state** that we did before. Predict overall ratings for the test set, and then compute and print out the root-mean-squared error for this model. You should find a root-mean-squared error of 3.94, only slightly worse than we got using the full set of 33 features. More features are not always that helpful!

In [97]:
X = dfclean[largestlist]  #extracts the actual data
X

,reactions,ball_control,gk_diving
0,47.0,49.0,6.0
1,47.0,49.0,6.0
2,47.0,49.0,6.0
3,46.0,48.0,5.0
4,46.0,48.0,5.0
...,...,...,...
183973,86.0,85.0,9.0
183974,74.0,86.0,9.0
183975,74.0,86.0,9.0
183976,69.0,91.0,9.0


In [73]:
### YOUR CODE HERE!

#Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=123)

#Train a new Linear Regression model
regressor_new = LinearRegression()
regressor_new.fit(X_train, y_train)

#Make predictions
y_pred = regressor_new.predict(X_test)

# Step 5: Compute and print RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

3.944044941419205
